In [ ]:
import re
import time

from synapseclient.extensions.curator import (
    create_record_based_metadata_task,
    create_file_based_metadata_task,
    query_schema_registry
)
from synapseclient.models import (
    File,
    Grid,
    RecordSet,
    ViewTypeMask,
)
from synapseclient import Synapse
import synapseutils
from synapseclient.models.table_components import Query

syn = Synapse()
syn.login()

In [ ]:
%pip install --upgrade synapseclient

## Browse Available Schemas
### List All Availble Schemas for AD

In [ ]:
# Find the latest schema for your specific data type
schema_uri = query_schema_registry(
    synapse_client=syn,
    dcc="ad",  # Your data coordination center, check out the `syn69735275` table if you do not know your code
    datatype="IndividualAnimalMetadataTemplate"  # Your specific data type
)

print("Latest schema URI:", schema_uri)

## All AD schemas

In [ ]:
# Find the latest schema for your specific dcc and/or data type
schema_uri = query_schema_registry(
    synapse_client=syn,
    dcc="ad",  # Your data coordination center, check out the `syn69735275` table if you do not know your code
    #datatype="IndividualAnimalMetadataTemplate"  # Your specific data type
)

print("Latest schema URI:", schema_uri)

## List Specific Schemas For This Study

In [ ]:
def get_latest_schema_uri(
    datatype: str,
    dcc: str = "ad",
    synapse_client: Synapse = None,
) -> str:
    """Return the latest registered schema URI for a data type.

    Args:
        datatype: Template name (e.g. "BiospecimenMetadataTemplate").
        dcc: Data coordination center code; check the `syn69735275` table if unknown.
        synapse_client: Logged-in Synapse client (defaults to the global `syn`).
    """
    uri = query_schema_registry(
        synapse_client=synapse_client or syn,
        dcc=dcc,
        datatype=datatype,
    )
    if not uri:
        raise ValueError(f"No schema found for dcc='{dcc}', datatype='{datatype}'")
    print(f"Latest schema URI for {datatype}: {uri}")
    return uri


INDIVIDUAL_TEMPLATES = {
    "human": "IndividualHumanMetadataTemplate",
    "animal": "IndividualAnimalMetadataTemplate",
    "model-ad": "IndividualAnimalMODEL.ADMetadataTemplate",
}

BIOSPECIMEN_TEMPLATES = {
    "biospecimen": "BiospecimenMetadataTemplate",
    "invitro": "BiospecimenInvitroMetadataTemplate",
    "drosophila": "BiospecimenDrosophilaMetadataTemplate",
}


def _get_template_uri(choice: str, templates: dict, label: str, **kwargs) -> str:
    """Look up `choice` (case-insensitive) in `templates` and return its latest schema URI."""
    key = choice.lower()
    if key not in templates:
        raise ValueError(f"Unknown {label} '{choice}'; choose one of {list(templates)}")
    return get_latest_schema_uri(templates[key], **kwargs)


def get_individual_schema_uri(individual_type: str, **kwargs) -> str:
    """Return the latest individual schema URI for "human", "animal", or "model-ad"."""
    return _get_template_uri(individual_type, INDIVIDUAL_TEMPLATES, "individual_type", **kwargs)


def get_biospecimen_schema_uri(biospecimen_type: str, **kwargs) -> str:
    """Return the latest biospecimen schema URI for "biospecimen", "invitro", or "drosophila"."""
    return _get_template_uri(biospecimen_type, BIOSPECIMEN_TEMPLATES, "biospecimen_type", **kwargs)


def get_assay_schema_uri(assay_type: str, **kwargs) -> str:
    """Return the latest assay schema URI, e.g. "ScrnaSeq" -> AssayScrnaSeqMetadataTemplate.

    Use the part of the template name between "Assay" and "MetadataTemplate"
    (see the "All AD schemas" output above for the full list).

    Notes:
        - assay_type must match the registry's capitalization (e.g. "ScrnaSeq",
          not "scrnaseq"); there is no fixed list of choices like the one used
          for individual templates.
        - An unrecognized assay_type raises ValueError ("No schema found ...").
        - MARMO.AD assay templates (e.g. MARMO.ADAssayProteomicsMetadataTemplate)
          use a different naming pattern and can't be reached here; call
          get_latest_schema_uri() with the full template name instead.
    """
    return get_latest_schema_uri(f"Assay{assay_type}MetadataTemplate", **kwargs)

## Put in the latest schemas you want to retrieve

In [ ]:
# Record-based metadata (data-specific)
INDIVIDUAL_TYPE = "human"  # "human", "animal", or "model-ad"
indiv_uri = get_individual_schema_uri(INDIVIDUAL_TYPE)
BIOSPECIMEN_TYPE = "biospecimen"  # "biospecimen", "invitro", or "drosophila"
biospec_uri = get_biospecimen_schema_uri(BIOSPECIMEN_TYPE)
ASSAY_TYPE = "ScrnaSeq"  # e.g. "ScrnaSeq", "RnaSeq", "Metabolomics", "Proteomics"
assay_uri = get_assay_schema_uri(ASSAY_TYPE)

# File-based metadata (not data-specific)
fileann_uri = get_latest_schema_uri("FileAnnotationTemplate")

## List Specific Schemas For This Study
## Section 3: Simple Curation Workflow
### Create Recoord-Based and File-Based Tasks (as administrator)
The Synapse Curator extension provides a Grid-based spreadsheet interface where human curators fill in metadata fields defined by the JSON schema. Two workflow options are available:

- **Option A — File-Based**: Creates an EntityView of all files in a staging folder; curators annotate files directly via a Grid.
- **Option B — Record-Based**: Metadata is stored as a RecordSet table; structured records are created by curators and annotations can be propagated to individual files.

> **Note**: The Curator extension provides a schema-driven UI for human curators to enter and validate data. For AI-assisted annotation, see `enhance_annotations_with_ai()` in `synapse_dataset_manager.py`.

In [ ]:
# Fill these in for your study
study_name = "<your-study-name>"
ad_backend_id = "synXXXXXXXX"         # Synapse project ID where the tasks will live
staging_folder_id = "synXXXXXXXX"     # Staging folder with the data files (file-based task)
metadata_folder_id = "synXXXXXXXX"    # Folder for the RecordSets (record-based tasks)
assay_folder_id = "synXXXXXXXX"       # Assay data folder
assignee_principal_id = None          # Optional: Synapse user or team ID to assign tasks to

### Record-Based Metadata Tasks

In [ ]:
##these are the record-based metadata schemas we will be using in this curation workflow
schema_uris = [
    indiv_uri,
    biospec_uri,
    assay_uri,
]   
schema_uris

### Create all Record-Based Metadata tasks at once

In [ ]:
# Keyed by the URIs selected above, so this follows INDIVIDUAL_TYPE / ASSAY_TYPE
upsert_key_map = {
    indiv_uri: ["individualID"],
    biospec_uri: ["specimenID"],
    assay_uri: ["specimenID"],
}
results = []
record_set_ids = {}  # schema_name -> RecordSet ID

for schema_uri in schema_uris:
    schema_name = re.search(r"ad-(.+?)-\d+\.\d+\.\d+", schema_uri).group(1)
    upsert_keys = upsert_key_map[schema_uri]

    try:
        record_set, curation_task, data_grid = create_record_based_metadata_task(
            synapse_client=syn,
            project_id=ad_backend_id,
            folder_id=metadata_folder_id,
            record_set_name=f"Test_{study_name}_{schema_name}",
            record_set_description=f"TEST: {study_name}_({schema_name})",
            curation_task_name=f"TEST_{study_name}_{schema_name}",
            upsert_keys=upsert_keys,
            instructions="Complete all required fields according to the schema.",
            schema_uri=schema_uri,
            bind_schema_to_record_set=True,
        )
        record_set_ids[schema_name] = record_set.id
        results.append({
            "schema_uri": schema_uri,
            "record_set": record_set,
            "curation_task": curation_task,
            "data_grid": data_grid,
        })
    except Exception as exc:
        results.append({"schema_uri": schema_uri, "error": str(exc)})
        print(f"[{schema_name}] FAILED: {exc}")

# Give Synapse time to finish setting up the new RecordSets before loading data
# (the Synapse curator docs do the same after creating a task)
time.sleep(10)

# RecordSet IDs (copy this dict to reuse in later cells)
print("\nRECORD_SET_IDS = {")
for name, rs_id in record_set_ids.items():
    print(f'    "{name}": "{rs_id}",')
print("}")

### File-Based Metadata Tasks

In [ ]:
# Derive the name from fileann_uri (not the leftover schema_name from the record-based loop)
fileann_name = re.search(r"ad-(.+?)-\d+\.\d+\.\d+", fileann_uri).group(1)

entity_view_id, task_id = create_file_based_metadata_task(
    synapse_client=syn,
    folder_id=staging_folder_id,     # Staging Folder
    curation_task_name=f"TEST_{study_name}_AllFiles_{fileann_name}",  # Must be unique within project
    instructions="Complete all required fields according to the schema.",
    attach_wiki=False,               # Creates a wiki in the folder with the entity view (Defaults to False)
    entity_view_name=f"TEST {study_name} File View",
    assignee_principal_id=assignee_principal_id,
    schema_uri=fileann_uri)

print(f"Created EntityView: {entity_view_id}")
print(f"Created CurationTask: {task_id}")

### Work with metadata and validate (Record-based workflow)
After creating a record-based metadata task, collaborators can enter metadata through the Grid interface. Once metadata entry is complete, you'll want to validate the data against your schema and identify any issues.
The metadata curation workflow

    Data Entry: Collaborators use the Grid interface (via the curation task link in the Synapse web UI) to enter metadata
    Grid Export: Export the Grid session back to the RecordSet to save changes (this can be done via the web UI or programmatically)
    Validation: Retrieve detailed validation results to identify schema violations
    Correction: Fix any validation errors and repeat as needed

Creating and exporting a Grid session

Validation results are only generated when a Grid session is exported back to the RecordSet. This triggers Synapse to validate each row against the bound schema. You have two options:

Option A: Via the Synapse web UI (most common)

Users can access the curation task through the Synapse web interface, enter/edit data in the Grid, and click the export button. This automatically generates validation results.

Option B: Programmatically create and export a Grid session

(option B below)

## Create grid for Record set IDs

In [ ]:
# Paste the RECORD_SET_IDS printed by the record-based task cell above
RECORD_SET_IDS = {
    "IndividualHumanMetadataTemplate": "synXXXXXXXX",
    "BiospecimenMetadataTemplate": "synXXXXXXXX",
    "AssayScrnaSeqMetadataTemplate": "synXXXXXXXX",
}
grids = {}  # keep these around so you can call .export_to_record_set() / .delete() on each later

for name, rs_id in RECORD_SET_IDS.items():
    record_set = RecordSet(id=rs_id).get(synapse_client=syn)
    grid = Grid(record_set_id=record_set.id).create(synapse_client=syn)
    grids[name] = grid
    print(f"[{name}] RecordSet {rs_id} -> Grid session {grid.session_id}")
    print(f"  https://www.synapse.org/Grid:default?sessionId={grid.session_id}")

In [ ]:
# Source metadata CSVs in the study's Metadata folder, keyed like RECORD_SET_IDS above
SOURCE_CSV_IDS = {
    "IndividualHumanMetadataTemplate": "synXXXXXXXX",  # individual metadata CSV
    "BiospecimenMetadataTemplate": "synXXXXXXXX",      # biospecimen metadata CSV
    "AssayScrnaSeqMetadataTemplate": "synXXXXXXXX",    # assay metadata CSV
}

validation_results = {}

for name, rs_id in RECORD_SET_IDS.items():
    source_csv_id = SOURCE_CSV_IDS[name]
    print(f"\n=== {name} ===")

    # --- Original approach (kept for bug reporting) ---
    # Point the RecordSet at the existing CSV's file handle -- no download/reupload.
    # Fails with 403 "Only the creator of a FileHandle can access it directly by its ID"
    # when someone else uploaded the CSV.
    # source_file = File(id=source_csv_id, download_file=False).get(synapse_client=syn)
    # record_set = RecordSet(id=rs_id).get(synapse_client=syn)
    # record_set.data_file_handle_id = source_file.data_file_handle_id
    # record_set = record_set.store(synapse_client=syn)
    # print(f"Pointed RecordSet {rs_id} at file handle {source_file.data_file_handle_id} ({source_csv_id})")

    # --- Workaround 1 (kept for reference): download the CSV and re-upload it ---
    # source_file = File(id=source_csv_id, download_location="source_csvs").get(synapse_client=syn)
    # record_set = RecordSet(id=rs_id).get(synapse_client=syn)
    # record_set.path = source_file.path
    # record_set = record_set.store(synapse_client=syn)
    # print(f"Uploaded {source_file.name} ({source_csv_id}) to RecordSet {rs_id}")

    # --- Workaround 2 (current): copy the file handle so you own it -- no download/reupload ---
    source_file = File(id=source_csv_id, download_file=False).get(synapse_client=syn)
    copy_result = synapseutils.copyFileHandles(
        syn,
        fileHandles=[source_file.data_file_handle_id],
        associateObjectTypes=["FileEntity"],
        associateObjectIds=[source_csv_id],
    )[0]
    if copy_result.get("failureCode"):
        raise RuntimeError(f"File handle copy failed for {source_csv_id}: {copy_result['failureCode']}")
    new_handle_id = copy_result["newFileHandle"]["id"]

    record_set = RecordSet(id=rs_id).get(synapse_client=syn)
    record_set.data_file_handle_id = new_handle_id
    record_set = record_set.store(synapse_client=syn)
    print(f"Pointed RecordSet {rs_id} at copied file handle {new_handle_id} (from {source_csv_id})")

    # Create + export a Grid session to trigger validation
    grid = Grid(record_set_id=record_set.id).create(synapse_client=syn)
    grid.export_to_record_set(synapse_client=syn)
    grid.delete(synapse_client=syn)

    # Refresh and pull validation results
    record_set = RecordSet(id=rs_id).get(synapse_client=syn)
    validation_df = record_set.get_detailed_validation_results(synapse_client=syn)

    if validation_df is None:
        print("No validation results yet.")
        validation_results[name] = None
        continue

    total = len(validation_df)
    valid = (validation_df["is_valid"] == True).sum()  # noqa: E712
    invalid = (validation_df["is_valid"] == False).sum()  # noqa: E712
    print(f"{valid}/{total} valid, {invalid} invalid")

    if invalid > 0:
        invalid_rows = validation_df[validation_df["is_valid"] == False]  # noqa: E712
        error_report_path = f"validation_errors_{name}.csv"
        invalid_rows[["row_index", "validation_error_message", "all_validation_messages"]].to_csv(error_report_path, index=False)
        print(f"Detailed error report saved to: {error_report_path}")
        for _, row in invalid_rows.head(3).iterrows():
            print(f"  row {row['row_index']}: {row['validation_error_message']}")

    validation_results[name] = validation_df

# Grid sessions opened before the data was loaded still show the old (empty) RecordSet
# and don't refresh. Close them and open fresh sessions that show the loaded data.
# Note: this discards any unsaved edits in those older sessions.
grids = {}
for name, rs_id in RECORD_SET_IDS.items():
    for old_grid in Grid.list(source_id=rs_id, synapse_client=syn):
        old_grid.delete(synapse_client=syn)
    grid = Grid(record_set_id=rs_id).create(synapse_client=syn)
    grids[name] = grid
    print(f"[{name}] RecordSet {rs_id} -> Grid session {grid.session_id}")
    print(f"  https://www.synapse.org/Grid:default?sessionId={grid.session_id}")